## 1. Set up

# This notebook outputs layer similarities to help decide which layers to trim
Code source for Part 3: https://github.com/arcee-ai/PruneMe/tree/main

In [ ]:
#load utils
from google.colab import files
uploaded = files.upload()  # This will open a file picker

Saving utils.py to utils.py


In [ ]:
!pip install -q transformers
!pip install -q torchinfo
!pip install -q datasets
!pip install -q evaluate
!pip install -q transformers[torch]
!pip install -q peft
!pip install -q accelerate
!pip install bitsandbytes
#!pip install -U bitsandbytes


import logging
import csv
import argparse
import numpy as np
from tqdm import tqdm
import pandas as pd

import torch
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from transformers import AutoModel, AutoModelForSequenceClassification
import datasets
from datasets import load_dataset

from utils import get_last_non_padded_tokens, compute_block_distances
from typing import Optional

logging.basicConfig(level=logging.INFO)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 7.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-cupti-cu12 12.5.82 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-nvrtc-cu12==12.

In [ ]:
import logging
import csv
import argparse
import numpy as np
from tqdm import tqdm


import torch
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from transformers import AutoModel, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
import datasets
from datasets import load_dataset

from utils import get_last_non_padded_tokens, compute_block_distances
from typing import Optional

logging.basicConfig(level=logging.INFO)

## 2. Load Data

In [ ]:
# Load the dataset
moral_dataset = load_dataset("demelin/moral_stories", "cls-action+context+consequence-norm_distance")

# Select only the first 4000 for training and first 1000 for testing
moral_train_dataset = moral_dataset['train'].shuffle().select(range(20000))
moral_dev_dataset = moral_dataset['test'].shuffle().select(range(2000))

README.md:   0%|          | 0.00/12.0k [00:00<?, ?B/s]

moral_stories.py:   0%|          | 0.00/11.4k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/3.70M [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/374k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/374k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
def preprocess_moral(data):
    merged_texts = []

    for moral, immoral in zip(data['moral_action'], data['immoral_action']):
        # Ignore "not specified" values for moral and immoral actions
        if moral.lower() == "not specified":
            action_text = immoral
        elif immoral.lower() == "not specified":
            action_text = moral
        else:
            action_text = f"{moral} {immoral}"  # Merge both sentences

        # Add the [CLS] token at the beginning, followed by situation, norm, and action_text, all separated by [SEP]
        #action_text = f"[CLS] {action_text} [SEP]"

        merged_texts.append(action_text)  # Store the merged text

    return merged_texts

In [ ]:
type(preprocess_moral(moral_train_dataset))

list

In [ ]:
MAX_SEQUENCE_LENGTH = 128

## 3. Main function for layer similarity calculation

In [ ]:
def main(model_path: str, dataset, batch_size: int, max_length: int,
         layers_to_skip: int, dataset_size: Optional[int] = None):

    device = "cuda" if torch.cuda.is_available() else "cpu"

    # if resource is a problem
    quantization_config = BitsAndBytesConfig(load_in_4bit=True,
                                            bnb_4bit_use_double_quant=True,
                                            bnb_4bit_quant_type="nf4",
                                            bnb_4bit_compute_dtype=torch.bfloat16)

    model = AutoModelForSequenceClassification.from_pretrained(model_path,
                                                 device_map="auto",
                                                 quantization_config=quantization_config,
                                                 output_hidden_states=True)

    tokenizer = AutoTokenizer.from_pretrained(model_path)

    if not tokenizer.pad_token:
        tokenizer.pad_token = tokenizer.eos_token

    model.eval()

    #dataset = datasets.load_dataset(dataset, split=dataset_subset)
    #if dataset_size:
    #    dataset = dataset.select(range(dataset_size))

    dataloader = DataLoader(preprocess_moral(dataset), batch_size=batch_size, shuffle=False, drop_last=True) # torch object

    # Initialize a list to store distances for each block across the dataset
    all_distances = [[] for _ in range(model.config.num_hidden_layers - layers_to_skip)]


    for batch in tqdm(dataloader, desc="Processing batches"):
        inputs = tokenizer(batch, return_tensors="pt", padding="longest", max_length=max_length, truncation=True).to(device)
        with torch.no_grad():
            outputs = model(**inputs)
        attention_mask = inputs["attention_mask"]
        hidden_states = outputs.hidden_states
        last_non_padded_hidden_states = get_last_non_padded_tokens(hidden_states, attention_mask)

        # Remove the first element to account for the input layer not being considered a model hidden layer
        # This adjustment is necessary for analyses focusing on the model's internal transformations
        last_non_padded_hidden_states = last_non_padded_hidden_states[1:]

        # Ensure that the length of last_non_padded_hidden_states matches the number of model hidden layers minus one
        assert len(last_non_padded_hidden_states) == model.config.num_hidden_layers, "Length of last_non_padded_hidden_states  \
        does not match expected number of hidden layers."

        # Compute distances and append to all_distances
        distances = compute_block_distances(last_non_padded_hidden_states, layers_to_skip)
        for i, distance in enumerate(distances):
            all_distances[i].append(distance)

    # Calculate average distances for each block
    average_distances = [np.mean(block_distances) for block_distances in all_distances]

    # Write the average distances to a CSV file and compute the minimum average distance
    min_distance = float('inf')  # Initialize with infinity
    min_distance_layer = 0  # Initialize with an impossible value
    data = []

    for i, avg_dist in enumerate(average_distances):
        data.append({
            'block_start': i + 1,  # layer indices are 1-based in the paper
            'block_end': i + 1 + layers_to_skip,
            'average_distance': avg_dist
        })
        if avg_dist < min_distance:
            min_distance = avg_dist
            min_distance_layer = i + 1

    return pd.DataFrame(data)


In [ ]:
df = main(model_path='roberta-large', dataset=moral_dev_dataset, batch_size=8, max_length=128, layers_to_skip=2, dataset_size=2000)

for i in range(4, 10,2):
  print("layer to skip " + str(i))
  new_df = main(model_path='roberta-large', dataset=moral_dev_dataset, batch_size=8, max_length=128, layers_to_skip=i, dataset_size=2000)
  df = pd.concat([df, new_df])

df

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

layer to skip 6


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Processing batches: 100%|██████████| 250/250 [00:25<00:00,  9.99it/s]


layer to skip 8


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Processing batches: 100%|██████████| 250/250 [00:27<00:00,  9.20it/s]


,block_start,block_end,average_distance
0,1,3,0.158395
1,2,4,0.176190
2,3,5,0.200081
3,4,6,0.147936
4,5,7,0.137458
...,...,...,...
11,12,20,0.343243
12,13,21,0.377365
13,14,22,0.433729
14,15,23,0.332913


## 4. Layer Prunning and Heal

In [ ]:
!pip install -q evaluate
import evaluate

In [ ]:
torch.manual_seed(2947)
np.random.seed(2947)

In [ ]:
# Load accuracy and F1 metrics
metric_acc = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=1)  # Convert logits to class predictions

    # Compute accuracy
    acc = metric_acc.compute(predictions=predictions, references=labels)

    # Compute F1-score (weighted to account for class imbalance)
    f1 = metric_f1.compute(predictions=predictions, references=labels, average="weighted")

    # Return both metrics
    return {"accuracy": acc["accuracy"], "f1": f1["f1"]}


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("roberta-large")

def trim_middle_layers_roberta(layer_start=8, layer_end=14):
    # Load model + tokenizer
    model = AutoModelForSequenceClassification.from_pretrained("roberta-large")

    total_layers = len(model.roberta.encoder.layer)

    # Validation
    if layer_start < 0 or layer_end > total_layers or layer_start > layer_end:
        raise ValueError(f"Invalid range: start={layer_start}, end={layer_end}, model has {total_layers} layers.")

    # Keep layers before and after the middle chunk
    new_layers = (
        model.roberta.encoder.layer[:layer_start] +  # Early layers
        model.roberta.encoder.layer[layer_end + 1:]  # Late layers
    )

    model.roberta.encoder.layer = new_layers

    # Update model config
    model.config.num_hidden_layers = len(new_layers)

    print(f"Trimmed out layers {layer_start} to {layer_end}. Remaining layers: {len(new_layers)}")
    return model

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:
def tokenize_function(examples, tokenizer):
    return tokenizer(preprocess_moral(examples),
                     padding="max_length", truncation=True, max_length=MAX_SEQUENCE_LENGTH)

In [ ]:
# Tokenize
train_dataset = moral_train_dataset.map(lambda x: tokenize_function(x, tokenizer), batched=True)
test_dataset = moral_dev_dataset.map(lambda x: tokenize_function(x, tokenizer), batched=True)

# Set correct columns
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])



Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
# number of layer trimmed = 6

results_list = []
layer_skip_from_list = []

for i in range(2, 18, 2):
  model = trim_middle_layers_roberta(layer_start=i, layer_end=i+6)

  training_args = TrainingArguments(
      output_dir="./results",
      evaluation_strategy="epoch",
      save_strategy="epoch",
      per_device_train_batch_size=16,
      per_device_eval_batch_size=16,
      num_train_epochs=1,
      weight_decay=0.05,
      learning_rate=2e-5,
      logging_dir="./logs",
      logging_steps=200
  )

  trainer = Trainer(
      model=model,
      args=training_args,
      train_dataset=train_dataset,
      eval_dataset=test_dataset,
      compute_metrics=compute_metrics
  )

  trainer.train()

  output = trainer.train()
  last_metrics = output.metrics
  layer_skip_from_list.append(i)
  results_list.append(last_metrics)


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Trimmed out layers 2 to 8. Remaining layers: 17


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.697700,0.693171,0.500000,0.333333


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.697400,0.693545,0.500000,0.333333


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


...

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.364200,0.422244,0.821000,0.820533


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Trimmed out layers 16 to 22. Remaining layers: 17


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.421200,0.435711,0.805500,0.805180


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.377200,0.430425,0.814000,0.813456


In [ ]:
results_list

[{'train_runtime': 1056.9139,
  'train_samples_per_second': 18.923,
  'train_steps_per_second': 1.183,
  'total_flos': 3305310812160000.0,
  'train_loss': 0.7008516784667969,
  'epoch': 1.0},
 {'train_runtime': 1057.0015,
  'train_samples_per_second': 18.921,
  'train_steps_per_second': 1.183,
  'total_flos': 3305310812160000.0,
  'train_loss': 0.6981867034912109,
  'epoch': 1.0},
 {'train_runtime': 1063.0308,
  'train_samples_per_second': 18.814,
  'train_steps_per_second': 1.176,
  'total_flos': 3305310812160000.0,
  'train_loss': 0.43346569213867187,
  'epoch': 1.0},
 {'train_runtime': 1060.4206,
  'train_samples_per_second': 18.86,
  'train_steps_per_second': 1.179,
  'total_flos': 3305310812160000.0,
  'train_loss': 0.6979152801513672,
  'epoch': 1.0},
 {'train_runtime': 1077.8434,
  'train_samples_per_second': 18.556,
  'train_steps_per_second': 1.16,
  'total_flos': 3305310812160000.0,
  'train_loss': 0.3771708969116211,
  'epoch': 1.0},
 {'train_runtime': 1063.6226,
  'train_sa

In [ ]:
layer_skip_from_list

[2, 4, 6, 8, 10, 12, 14, 16]

In [ ]:
# number of layer trimmed = 6

results_list = []
layer_skip_from_list = []

for i in range(18, 20, 2):
  model = trim_middle_layers_roberta(layer_start=i, layer_end=i+6)

  training_args = TrainingArguments(
      output_dir="./results",
      evaluation_strategy="epoch",
      save_strategy="epoch",
      per_device_train_batch_size=16,
      per_device_eval_batch_size=16,
      num_train_epochs=1,
      weight_decay=0.05,
      learning_rate=2e-5,
      logging_dir="./logs",
      logging_steps=200
  )

  trainer = Trainer(
      model=model,
      args=training_args,
      train_dataset=train_dataset,
      eval_dataset=test_dataset,
      compute_metrics=compute_metrics
  )

  trainer.train()

  output = trainer.train()
  last_metrics = output.metrics
  layer_skip_from_list.append(i)
  results_list.append(last_metrics)


model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Trimmed out layers 18 to 24. Remaining layers: 18


wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: statmathcat (statmathcat-university-of-california-berkeley) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.362500,0.421042,0.827000,0.826311


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.338700,0.412228,0.830000,0.829700


In [ ]:
# number of layer trimmed = 8

results_list = []
layer_skip_from_list = []

for i in range(2, 18, 2):
  model = trim_middle_layers_roberta(layer_start=i, layer_end=i+8)

  training_args = TrainingArguments(
      output_dir="./results",
      evaluation_strategy="epoch",
      save_strategy="epoch",
      per_device_train_batch_size=16,
      per_device_eval_batch_size=16,
      num_train_epochs=1,
      weight_decay=0.05,
      learning_rate=2e-5,
      logging_dir="./logs",
      logging_steps=200
  )

  trainer = Trainer(
      model=model,
      args=training_args,
      train_dataset=train_dataset,
      eval_dataset=test_dataset,
      compute_metrics=compute_metrics
  )

  output = trainer.train()

  #output = trainer.train()
  last_metrics = output.metrics
  layer_skip_from_list.append(i)
  results_list.append(last_metrics)


model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Trimmed out layers 2 to 10. Remaining layers: 15


wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

...

Trimmed out layers 14 to 22. Remaining layers: 15


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.408900,0.439530,0.812000,0.811728


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Trimmed out layers 16 to 24. Remaining layers: 16


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.389100,0.427284,0.821000,0.820620


In [ ]:
# number of layer trimmed = 10

results_list = []
layer_skip_from_list = []

for i in range(2, 26-10, 2):
  model = trim_middle_layers_roberta(layer_start=i, layer_end=i+10)

  training_args = TrainingArguments(
      output_dir="./results",
      evaluation_strategy="epoch",
      save_strategy="epoch",
      per_device_train_batch_size=16,
      per_device_eval_batch_size=16,
      num_train_epochs=1,
      weight_decay=0.05,
      learning_rate=2e-5,
      logging_dir="./logs",
      logging_steps=200
  )

  trainer = Trainer(
      model=model,
      args=training_args,
      train_dataset=train_dataset,
      eval_dataset=test_dataset,
      compute_metrics=compute_metrics
  )

  output = trainer.train()

  #output = trainer.train()
  last_metrics = output.metrics
  layer_skip_from_list.append(i)
  results_list.append(last_metrics)


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Trimmed out layers 2 to 12. Remaining layers: 13


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.570100,0.591568,0.714000,0.713997


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Trimmed out layers 4 to 14. Remaining layers: 13


...

Trimmed out layers 10 to 20. Remaining layers: 13


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.444500,0.477783,0.793500,0.793209


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Trimmed out layers 12 to 22. Remaining layers: 13


Epoch,Training Loss,Validation Loss


In [ ]:
for i in range(12, 26-10, 2):
  model = trim_middle_layers_roberta(layer_start=i, layer_end=i+10)

  training_args = TrainingArguments(
      output_dir="./results",
      evaluation_strategy="epoch",
      save_strategy="epoch",
      per_device_train_batch_size=16,
      per_device_eval_batch_size=16,
      num_train_epochs=1,
      weight_decay=0.05,
      learning_rate=2e-5,
      logging_dir="./logs",
      logging_steps=200
  )

  trainer = Trainer(
      model=model,
      args=training_args,
      train_dataset=train_dataset,
      eval_dataset=test_dataset,
      compute_metrics=compute_metrics
  )

  output = trainer.train()
  last_metrics = output.metrics



Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Trimmed out layers 12 to 22. Remaining layers: 13


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: statmathcat (statmathcat-university-of-california-berkeley) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.415100,0.453224,0.809500,0.808823


NameError: name 'layer_skip_from_list' is not defined

In [ ]:
for i in range(14, 26-10, 2):
  model = trim_middle_layers_roberta(layer_start=i, layer_end=i+10)

  training_args = TrainingArguments(
      output_dir="./results",
      evaluation_strategy="epoch",
      save_strategy="epoch",
      per_device_train_batch_size=16,
      per_device_eval_batch_size=16,
      num_train_epochs=1,
      weight_decay=0.05,
      learning_rate=2e-5,
      logging_dir="./logs",
      logging_steps=200
  )

  trainer = Trainer(
      model=model,
      args=training_args,
      train_dataset=train_dataset,
      eval_dataset=test_dataset,
      compute_metrics=compute_metrics
  )

  output = trainer.train()
  last_metrics = output.metrics

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Trimmed out layers 14 to 24. Remaining layers: 14


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.407000,0.440050,0.810000,0.809505


In [ ]:
# number of layer trimmed = 12

results_list = []
layer_skip_from_list = []

for i in range(2, 26-12, 2):
  model = trim_middle_layers_roberta(layer_start=i, layer_end=i+12)

  training_args = TrainingArguments(
      output_dir="./results",
      evaluation_strategy="epoch",
      save_strategy="epoch",
      per_device_train_batch_size=16,
      per_device_eval_batch_size=16,
      num_train_epochs=1,
      weight_decay=0.05,
      learning_rate=2e-5,
      logging_dir="./logs",
      logging_steps=200
  )

  trainer = Trainer(
      model=model,
      args=training_args,
      train_dataset=train_dataset,
      eval_dataset=test_dataset,
      compute_metrics=compute_metrics
  )

  output = trainer.train()
  last_metrics = output.metrics


model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Trimmed out layers 2 to 14. Remaining layers: 11


wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

...

Trimmed out layers 10 to 22. Remaining layers: 11


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.448800,0.465549,0.799000,0.798753


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Trimmed out layers 12 to 24. Remaining layers: 12


Epoch,Training Loss,Validation Loss


In [ ]:
for i in range(12, 26-12, 2):
  model = trim_middle_layers_roberta(layer_start=i, layer_end=i+12)

  training_args = TrainingArguments(
      output_dir="./results",
      evaluation_strategy="epoch",
      save_strategy="epoch",
      per_device_train_batch_size=16,
      per_device_eval_batch_size=16,
      num_train_epochs=1,
      weight_decay=0.05,
      learning_rate=2e-5,
      logging_dir="./logs",
      logging_steps=200
  )

  trainer = Trainer(
      model=model,
      args=training_args,
      train_dataset=train_dataset,
      eval_dataset=test_dataset,
      compute_metrics=compute_metrics
  )

  output = trainer.train()
  last_metrics = output.metrics


model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Trimmed out layers 12 to 24. Remaining layers: 12


wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: statmathcat (statmathcat-university-of-california-berkeley) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.416300,0.458400,0.803000,0.802564


In [ ]:
# number of layer trimmed = 14

for i in range(2, 26-14, 2):
  model = trim_middle_layers_roberta(layer_start=i, layer_end=i+14)

  training_args = TrainingArguments(
      output_dir="./results",
      evaluation_strategy="epoch",
      save_strategy="epoch",
      per_device_train_batch_size=16,
      per_device_eval_batch_size=16,
      num_train_epochs=1,
      weight_decay=0.05,
      learning_rate=2e-5,
      logging_dir="./logs",
      logging_steps=200
  )

  trainer = Trainer(
      model=model,
      args=training_args,
      train_dataset=train_dataset,
      eval_dataset=test_dataset,
      compute_metrics=compute_metrics
  )

  output = trainer.train()
  last_metrics = output.metrics


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Trimmed out layers 2 to 16. Remaining layers: 9


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.527900,0.572225,0.728000,0.727564


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Trimmed out layers 4 to 18. Remaining layers: 9


...

Trimmed out layers 8 to 22. Remaining layers: 9


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.447900,0.491400,0.772000,0.771537


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Trimmed out layers 10 to 24. Remaining layers: 10


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.432400,0.479841,0.790000,0.789593


In [ ]:
# number of layer trimmed = 16

for i in range(2, 26-16, 2):
  model = trim_middle_layers_roberta(layer_start=i, layer_end=i+16)

  training_args = TrainingArguments(
      output_dir="./results",
      evaluation_strategy="epoch",
      save_strategy="epoch",
      per_device_train_batch_size=16,
      per_device_eval_batch_size=16,
      num_train_epochs=1,
      weight_decay=0.05,
      learning_rate=2e-5,
      logging_dir="./logs",
      logging_steps=200
  )

  trainer = Trainer(
      model=model,
      args=training_args,
      train_dataset=train_dataset,
      eval_dataset=test_dataset,
      compute_metrics=compute_metrics
  )

  output = trainer.train()
  last_metrics = output.metrics


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Trimmed out layers 2 to 18. Remaining layers: 7


wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: statmathcat (statmathcat-university-of-california-berkeley) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.547000,0.558985,0.721500,0.721223


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


...

Trimmed out layers 6 to 22. Remaining layers: 7


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.488100,0.514303,0.764000,0.763176


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Trimmed out layers 8 to 24. Remaining layers: 8


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.465300,0.488251,0.780000,0.779426


In [ ]:
# number of layer trimmed = 18

for i in range(2, 26-18, 2):
  model = trim_middle_layers_roberta(layer_start=i, layer_end=i+18)

  training_args = TrainingArguments(
      output_dir="./results",
      evaluation_strategy="epoch",
      save_strategy="epoch",
      per_device_train_batch_size=16,
      per_device_eval_batch_size=16,
      num_train_epochs=1,
      weight_decay=0.05,
      learning_rate=2e-5,
      logging_dir="./logs",
      logging_steps=200
  )

  trainer = Trainer(
      model=model,
      args=training_args,
      train_dataset=train_dataset,
      eval_dataset=test_dataset,
      compute_metrics=compute_metrics
  )

  output = trainer.train()
  last_metrics = output.metrics


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Trimmed out layers 2 to 20. Remaining layers: 5


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.536600,0.574823,0.728500,0.727985


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Trimmed out layers 4 to 22. Remaining layers: 5


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.527700,0.548589,0.738000,0.737468


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Trimmed out layers 6 to 24. Remaining layers: 6


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.517300,0.544977,0.737000,0.736112


In [ ]:
# number of layer trimmed = 2 and 4, only prunning the very last layer to save compute time
for i in [2,4]:
  model = trim_middle_layers_roberta(layer_start=24-i, layer_end=24)

  training_args = TrainingArguments(
      output_dir="./results",
      evaluation_strategy="epoch",
      save_strategy="epoch",
      per_device_train_batch_size=16,
      per_device_eval_batch_size=16,
      num_train_epochs=1,
      weight_decay=0.05,
      learning_rate=2e-5,
      logging_dir="./logs",
      logging_steps=200
  )

  trainer = Trainer(
      model=model,
      args=training_args,
      train_dataset=train_dataset,
      eval_dataset=test_dataset,
      compute_metrics=compute_metrics
  )

  output = trainer.train()
  last_metrics = output.metrics

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Trimmed out layers 22 to 24. Remaining layers: 22


wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

...

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: statmathcat (statmathcat-university-of-california-berkeley) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.394200,0.423332,0.823000,0.822382


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Trimmed out layers 20 to 24. Remaining layers: 20


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.379800,0.400478,0.839000,0.838613


## 5. Getting inference time

In [ ]:
def preprocess_moral(data, tokenizer):
    merged_texts = []

    for moral, immoral in zip(data['moral_action'], data['immoral_action']):
        # Ignore "not specified" values for moral and immoral actions
        if moral.lower() == "not specified":
            action_text = immoral
        elif immoral.lower() == "not specified":
            action_text = moral
        else:
            action_text = f"{moral} {immoral}"  # Merge both sentences

        # Add the [CLS] token at the beginning, followed by situation, norm, and action_text, all separated by [SEP]
        #action_text = f"[CLS] {action_text} [SEP]"

        merged_texts.append(action_text)  # Store the merged text

    # Tokenize the batch of merged sentences
    encoded = tokenizer.batch_encode_plus(
        merged_texts,
        max_length=MAX_SEQUENCE_LENGTH,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_token_type_ids=True,
        return_tensors="pt"
    )

    return {
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
        "token_type_ids": encoded["token_type_ids"]
    }


In [ ]:
import torch
import time
from torch.utils.data import Subset, DataLoader
import random

# Prepare the training and validation data
def preprocess_data(dataset, tokenizer):
    # Preprocess the data and tokenize it
    return dataset.map(preprocess_moral, batched=True, fn_kwargs={'tokenizer': tokenizer})
# Custom collate function to convert batch items to tensors
def collate_fn(batch):
    # Stack the batch and ensure it's in tensor format
    input_ids = torch.stack([torch.tensor(item['input_ids']) for item in batch])
    attention_mask = torch.stack([torch.tensor(item['attention_mask']) for item in batch])
    token_type_ids = torch.stack([torch.tensor(item['token_type_ids']) for item in batch])
    labels = torch.tensor([item['labels'] for item in batch])
    return {'input_ids': input_ids, 'attention_mask': attention_mask, 'token_type_ids': token_type_ids, 'labels': labels}
# Format dataset to match model input expectations
def format_dataset(example):
    return {
        "input_ids": example["input_ids"],
        "attention_mask": example["attention_mask"],
        "token_type_ids": example["token_type_ids"],
        "labels": example["label"] if "label" in example else 0  # Default to 0 if missing
    }
# Function to measure inference time on N samples
def measure_inference_time(model, dataloader, device, sample_size=100):
    model.eval()
    # Sample random examples from the dataset
    indices = random.sample(range(len(dataloader.dataset)), sample_size)
    sampled_dataset = Subset(dataloader.dataset, indices)
    sampled_dataloader = DataLoader(sampled_dataset, batch_size=8, collate_fn=collate_fn)
    # Warm-up (optional but good practice on GPU)
    for _ in range(5):
        for batch in sampled_dataloader:
            with torch.no_grad():
                model(batch['input_ids'].to(device),
                      attention_mask=batch['attention_mask'].to(device))
    # Accurate timing
    torch.cuda.synchronize()
    start_time = time.time()
    for batch in sampled_dataloader:
        with torch.no_grad():
            model(batch['input_ids'].to(device),
                  attention_mask=batch['attention_mask'].to(device))
    torch.cuda.synchronize()
    total_time = time.time() - start_time
    avg_time_per_batch = total_time / len(sampled_dataloader)
    print(f"\nModel: {model.__class__.__name__}")
    print(f"Total inference time on {sample_size} samples: {total_time:.4f} seconds")
    print(f"Average inference time per batch: {avg_time_per_batch:.4f} seconds")
    return total_time


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_path = "roberta-large"
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.to(device)
tokenizer = AutoTokenizer.from_pretrained(model_path)
formatted_dev_data = preprocess_data(moral_dev_dataset, tokenizer).map(format_dataset, remove_columns=moral_dev_dataset.column_names)
dev_dataloader = DataLoader(formatted_dev_data, batch_size=8, collate_fn=collate_fn)


# Example usage — test inference time for student model only
inference_time = measure_inference_time(model, dev_dataloader, device, sample_size=500)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total Parameters: {total_params:,}")

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]


Model: RobertaForSequenceClassification
Total inference time on 500 samples: 4.9158 seconds
Average inference time per batch: 0.0780 seconds
Total Parameters: 355,361,794


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
for i in range(2, 20, 2):
  print("./roberta_large_"+str(24-i)+"_layers")
  model = trim_middle_layers_roberta(layer_start=24-i, layer_end=24)
  model.to(device)
  inference_time = measure_inference_time(model, dev_dataloader, device, sample_size=500)
  total_params = sum(p.numel() for p in model.parameters())
  print(f"Total Parameters: {total_params:,}")

./roberta_large_22_layers


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trimmed out layers 22 to 24. Remaining layers: 22

Model: RobertaForSequenceClassification
Total inference time on 500 samples: 4.7508 seconds
Average inference time per batch: 0.0754 seconds
Total Parameters: 330,169,346
./roberta_large_20_layers


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trimmed out layers 20 to 24. Remaining layers: 20

Model: RobertaForSequenceClassification
Total inference time on 500 samples: 4.2387 seconds
Average inference time per batch: 0.0673 seconds
Total Parameters: 304,976,898
./roberta_large_18_layers


...

Trimmed out layers 10 to 24. Remaining layers: 10

Model: RobertaForSequenceClassification
Total inference time on 500 samples: 2.1225 seconds
Average inference time per batch: 0.0337 seconds
Total Parameters: 179,014,658
./roberta_large_8_layers


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trimmed out layers 8 to 24. Remaining layers: 8

Model: RobertaForSequenceClassification
Total inference time on 500 samples: 1.7367 seconds
Average inference time per batch: 0.0276 seconds
Total Parameters: 153,822,210
./roberta_large_6_layers


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trimmed out layers 6 to 24. Remaining layers: 6

Model: RobertaForSequenceClassification
Total inference time on 500 samples: 1.2578 seconds
Average inference time per batch: 0.0200 seconds
Total Parameters: 128,629,762
